# 03 Reference baseline (Experiment A)

**Question.** What does the reference regression achieve on the frozen validation partition?

**Type.** Formal run or inspection of Experiment A ([spec](../docs/experiments/a-reference.md)). Does not train unless `RUN_TRAINING = True`. No official test data.

In [ ]:
import json, sys
from pathlib import Path
REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
import os; os.chdir(REPO)
import numpy as np
from ct_iqa.config.loader import load_study_config
from ct_iqa.experiment.runs import latest_completed_run, read_predictions
CONFIG = "configs/experiments/a_reference.yaml"
cfg = load_study_config(CONFIG)
print(cfg.run.experiment_id, cfg.model.name, cfg.loss.name, "bn_eps =", cfg.model.bn_eps)

## Run (explicit, expensive)
Set `RUN_TRAINING = True` only with an approved compute budget; this trains for the configured number of epochs and writes a new immutable run directory.

In [ ]:
RUN_TRAINING = False
DEVICE = "cpu"   # use "cuda" when available
if RUN_TRAINING:
    import dataclasses
    from ct_iqa.experiment.run import run_experiment
    cfg = dataclasses.replace(cfg, train=dataclasses.replace(cfg.train, device=DEVICE))
    run_dir = run_experiment(cfg, config_path=CONFIG)
else:
    run_dir = latest_completed_run(cfg.run.experiment_id)
print("run directory:", run_dir if run_dir else "NO COMPLETED RUN FOUND (status PENDING)")

In [ ]:
if run_dir:
    print(json.loads((run_dir / "metrics_validation.json").read_text()))
    hist = json.loads((run_dir / "history.json").read_text())
    print("selected epoch (1-based):", hist["best_epoch"], "policy:", hist["selection_policy"])

In [ ]:
if run_dir:
    import matplotlib.pyplot as plt
    ep = hist["epochs"]
    plt.plot([e["epoch"] for e in ep], [e["train_loss"] for e in ep], label="train loss")
    plt.plot([e["epoch"] for e in ep], [e["val_loss"] for e in ep], label="val loss"); plt.legend(); plt.xlabel("epoch (1-based)"); plt.show()